In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, a fictional lunar city. \n\n## SUMMARY\nKey information extracted includes:\n- The capital of the moon is Lunapolis.\n- The weather in Lunapolis is clear, with temperatures ranging from a high of 120C to a low of -100C.\n- There are 100,000 cheese miners residing in Lunapolis.\n- The cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f4ce5550-5b49-4f22-9beb-1e77b2a6c9cb'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='cde8ce6b-3954-4b99-b9e0-d6a161d52e26'),
              AIMessage(content='If I were Lunapolis’ new president, I’d pursue

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, a fictional lunar city. 

## SUMMARY
Key information extracted includes:
- The capital of the moon is Lunapolis.
- The weather in Lunapolis is clear, with temperatures ranging from a high of 120C to a low of -100C.
- There are 100,000 cheese miners residing in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5e44c3a5-9156-41b4-bc62-8bc66511c588'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='71e85c96-c1ec-4bda-a5fd-4aebdf1abf7f', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='21f0b4ea-c704-46c1-b4cf-a4abf2e6b641'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='8ba00ae4-6c30-4b87-be49-3359acd5f82d', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='0683747d-fd02-4927-9272-5a9e9139a5a6'),
              AIMessage(content='I can’t read the exact temperature of your device from here. I

In [8]:
print(response["messages"][-1].content)

I can’t read the exact temperature of your device from here. If you can tell me the type/model (phone, laptop, tablet, desktop) and any indicators you’re seeing, I can tailor steps. In the meantime, here’s how to handle a device that won’t turn on and how to check for overheating concerns:

If it feels hot or you suspect overheating
- Do not continue trying to power it on. If it’s hot, unplug from power (if possible) and let it cool in a safe place.
- Check for signs of damage: a burning smell, melted/soft plastic, or a swollen battery. If you see any of these, power it off and seek professional help immediately.
- When cool, try the normal power-on steps again.

General troubleshooting (works for many devices)
- Use the original power adapter and a known good outlet. Inspect the charger and cable for damage.
- If the device has a removable battery, reseat or remove the battery, then reconnect power and try to turn it on.
- Perform a forced shutdown: hold the power button for 15–30 sec